In [1]:
from pathlib import Path
import textwrap
import json
import requests

In [2]:
PROYECTO_DIR = Path("dia_17_api_sqlite")
PROYECTO_DIR.mkdir(exist_ok=True)

DATA_DIR = PROYECTO_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

print("Carpeta del proyecto:")
print(PROYECTO_DIR.resolve())

print("\nCarpeta de base de datos:")
print(DATA_DIR.resolve())

Carpeta del proyecto:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\dia_17_api_sqlite

Carpeta de base de datos:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\dia_17_api_sqlite\data


In [3]:
requirements = """
fastapi
uvicorn[standard]
requests
"""

(PROYECTO_DIR / "requirements.txt").write_text(requirements.strip(), encoding="utf-8")

print("requirements.txt creado correctamente.")

requirements.txt creado correctamente.


In [4]:
codigo_database = r'''
from pathlib import Path
import sqlite3


BASE_DIR = Path(__file__).resolve().parent
DATA_DIR = BASE_DIR / "data"
DB_PATH = DATA_DIR / "ventas.db"


def obtener_conexion():
    """
    Crea una conexión con SQLite.

    Buenas prácticas:
    - Crea la carpeta data si no existe.
    - Activa llaves foráneas.
    - Permite acceder a columnas por nombre.
    """
    DATA_DIR.mkdir(exist_ok=True)

    conexion = sqlite3.connect(DB_PATH)
    conexion.execute("PRAGMA foreign_keys = ON")
    conexion.row_factory = sqlite3.Row

    return conexion


def consultar_todos(sql, parametros=()):
    """
    Ejecuta una consulta SELECT y devuelve varias filas como diccionarios.
    """
    conexion = obtener_conexion()
    cursor = conexion.cursor()

    try:
        cursor.execute(sql, parametros)
        filas = cursor.fetchall()
        return [dict(fila) for fila in filas]

    finally:
        conexion.close()


def consultar_uno(sql, parametros=()):
    """
    Ejecuta una consulta SELECT y devuelve una sola fila como diccionario.
    """
    conexion = obtener_conexion()
    cursor = conexion.cursor()

    try:
        cursor.execute(sql, parametros)
        fila = cursor.fetchone()

        if fila is None:
            return None

        return dict(fila)

    finally:
        conexion.close()


def ejecutar(sql, parametros=()):
    """
    Ejecuta INSERT, UPDATE o DELETE.

    Devuelve:
    - ultimo_id
    - filas_afectadas
    """
    conexion = obtener_conexion()
    cursor = conexion.cursor()

    try:
        cursor.execute(sql, parametros)
        conexion.commit()

        return {
            "ultimo_id": cursor.lastrowid,
            "filas_afectadas": cursor.rowcount
        }

    except Exception:
        conexion.rollback()
        raise

    finally:
        conexion.close()


def ejecutar_varios(sql, lista_parametros):
    """
    Ejecuta una consulta varias veces.
    """
    conexion = obtener_conexion()
    cursor = conexion.cursor()

    try:
        cursor.executemany(sql, lista_parametros)
        conexion.commit()

        return {
            "filas_afectadas": cursor.rowcount
        }

    except Exception:
        conexion.rollback()
        raise

    finally:
        conexion.close()


def ejecutar_script(sql_script):
    """
    Ejecuta un bloque grande de SQL.
    """
    conexion = obtener_conexion()
    cursor = conexion.cursor()

    try:
        cursor.executescript(sql_script)
        conexion.commit()

    except Exception:
        conexion.rollback()
        raise

    finally:
        conexion.close()


def ejecutar_transaccion(funcion_operacion):
    """
    Ejecuta una operación compleja dentro de una transacción.

    La función recibida debe aceptar un cursor como argumento.

    Si todo sale bien:
        commit

    Si algo falla:
        rollback
    """
    conexion = obtener_conexion()
    cursor = conexion.cursor()

    try:
        cursor.execute("BEGIN")
        resultado = funcion_operacion(cursor)
        conexion.commit()
        return resultado

    except Exception:
        conexion.rollback()
        raise

    finally:
        conexion.close()
'''

(PROYECTO_DIR / "database.py").write_text(codigo_database, encoding="utf-8")

print("database.py creado correctamente.")

database.py creado correctamente.


In [5]:
parte_services_1 = r'''
import sqlite3
from datetime import datetime

from database import (
    consultar_todos,
    consultar_uno,
    ejecutar,
    ejecutar_varios,
    ejecutar_script,
    ejecutar_transaccion
)


SQL_CREAR_TABLAS = """
CREATE TABLE IF NOT EXISTS clientes (
    id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    correo TEXT NOT NULL UNIQUE,
    telefono TEXT,
    ciudad TEXT DEFAULT 'No especificada',
    activo INTEGER NOT NULL DEFAULT 1 CHECK(activo IN (0, 1))
);

CREATE TABLE IF NOT EXISTS productos (
    id_producto INTEGER PRIMARY KEY AUTOINCREMENT,
    sku TEXT NOT NULL UNIQUE,
    nombre TEXT NOT NULL,
    categoria TEXT NOT NULL,
    precio REAL NOT NULL CHECK(precio > 0),
    stock INTEGER NOT NULL DEFAULT 0 CHECK(stock >= 0),
    activo INTEGER NOT NULL DEFAULT 1 CHECK(activo IN (0, 1))
);

CREATE TABLE IF NOT EXISTS ventas (
    id_venta INTEGER PRIMARY KEY AUTOINCREMENT,
    id_cliente INTEGER NOT NULL,
    fecha TEXT NOT NULL,
    total REAL NOT NULL DEFAULT 0 CHECK(total >= 0),
    estado TEXT NOT NULL DEFAULT 'ACTIVA' CHECK(estado IN ('ACTIVA', 'CANCELADA')),

    FOREIGN KEY (id_cliente)
        REFERENCES clientes(id_cliente)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
);

CREATE TABLE IF NOT EXISTS detalle_ventas (
    id_detalle INTEGER PRIMARY KEY AUTOINCREMENT,
    id_venta INTEGER NOT NULL,
    id_producto INTEGER NOT NULL,
    cantidad INTEGER NOT NULL CHECK(cantidad > 0),
    precio_unitario REAL NOT NULL CHECK(precio_unitario > 0),
    subtotal REAL NOT NULL CHECK(subtotal >= 0),

    FOREIGN KEY (id_venta)
        REFERENCES ventas(id_venta)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,

    FOREIGN KEY (id_producto)
        REFERENCES productos(id_producto)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
);
"""
'''

In [6]:
parte_services_2 = r'''


CLIENTES_INICIALES = [
    ("Ana López", "ana@example.com", "555-111-2222", "CDMX"),
    ("Carlos Pérez", "carlos@example.com", "555-333-4444", "Guadalajara"),
    ("María Torres", "maria@example.com", None, "Monterrey"),
    ("Luis Romero", "luis@example.com", "555-888-9999", "Puebla")
]


PRODUCTOS_INICIALES = [
    ("P001", "Mouse inalámbrico", "Accesorios", 249.90, 15),
    ("P002", "Teclado mecánico", "Accesorios", 899.00, 8),
    ("P003", "Monitor 24 pulgadas", "Pantallas", 3299.00, 4),
    ("P004", "Cable HDMI", "Cables", 120.00, 20),
    ("P005", "Memoria USB 64GB", "Almacenamiento", 150.00, 30),
    ("P006", "Bocinas Bluetooth", "Audio", 699.00, 10)
]
'''

In [7]:
parte_services_3 = r'''


def limpiar_texto(valor):
    if valor is None:
        return ""

    return str(valor).strip()


def normalizar_sku(sku):
    return limpiar_texto(sku).upper()


def normalizar_correo(correo):
    return limpiar_texto(correo).lower()


def validar_texto_obligatorio(valor, nombre_campo):
    valor_limpio = limpiar_texto(valor)

    if valor_limpio == "":
        raise ValueError(f"El campo {nombre_campo} no puede estar vacío.")

    return valor_limpio


def validar_precio(precio):
    try:
        precio = float(precio)
    except (TypeError, ValueError):
        raise ValueError("El precio debe ser numérico.")

    if precio <= 0:
        raise ValueError("El precio debe ser mayor que cero.")

    return precio


def validar_stock(stock):
    try:
        stock = int(stock)
    except (TypeError, ValueError):
        raise ValueError("El stock debe ser un número entero.")

    if stock < 0:
        raise ValueError("El stock no puede ser negativo.")

    return stock


def validar_cantidad(cantidad):
    try:
        cantidad = int(cantidad)
    except (TypeError, ValueError):
        raise ValueError("La cantidad debe ser un número entero.")

    if cantidad <= 0:
        raise ValueError("La cantidad debe ser mayor que cero.")

    return cantidad
'''

In [8]:
parte_services_4 = r'''


def inicializar_base_datos():
    """
    Crea las tablas e inserta datos iniciales si no existen.
    """
    ejecutar_script(SQL_CREAR_TABLAS)

    ejecutar_varios("""
    INSERT OR IGNORE INTO clientes (nombre, correo, telefono, ciudad, activo)
    VALUES (?, ?, ?, ?, 1)
    """, CLIENTES_INICIALES)

    ejecutar_varios("""
    INSERT OR IGNORE INTO productos (sku, nombre, categoria, precio, stock, activo)
    VALUES (?, ?, ?, ?, ?, 1)
    """, PRODUCTOS_INICIALES)

    return {
        "ok": True,
        "mensaje": "Base de datos inicializada correctamente."
    }
'''

In [9]:
parte_services_5 = r'''


def listar_productos(solo_activos=True):
    if solo_activos:
        return consultar_todos("""
        SELECT
            id_producto,
            sku,
            nombre,
            categoria,
            precio,
            stock,
            activo
        FROM productos
        WHERE activo = 1
        ORDER BY nombre
        """)

    return consultar_todos("""
    SELECT
        id_producto,
        sku,
        nombre,
        categoria,
        precio,
        stock,
        activo
    FROM productos
    ORDER BY nombre
    """)


def buscar_producto_por_sku(sku):
    sku = normalizar_sku(sku)

    return consultar_uno("""
    SELECT
        id_producto,
        sku,
        nombre,
        categoria,
        precio,
        stock,
        activo
    FROM productos
    WHERE sku = ?
    """, (sku,))
'''

In [10]:
parte_services_6 = r'''


def registrar_producto(sku, nombre, categoria, precio, stock):
    try:
        sku = normalizar_sku(sku)
        nombre = validar_texto_obligatorio(nombre, "nombre")
        categoria = validar_texto_obligatorio(categoria, "categoría")
        precio = validar_precio(precio)
        stock = validar_stock(stock)

        resultado = ejecutar("""
        INSERT INTO productos (sku, nombre, categoria, precio, stock, activo)
        VALUES (?, ?, ?, ?, ?, 1)
        """, (sku, nombre, categoria, precio, stock))

        producto = buscar_producto_por_sku(sku)

        return {
            "ok": True,
            "mensaje": "Producto registrado correctamente.",
            "producto": producto,
            "id_producto": resultado["ultimo_id"]
        }

    except sqlite3.IntegrityError:
        return {
            "ok": False,
            "tipo_error": "conflicto",
            "mensaje": "Ya existe un producto con ese SKU."
        }

    except ValueError as error:
        return {
            "ok": False,
            "tipo_error": "validacion",
            "mensaje": str(error)
        }


def actualizar_producto(sku, datos_actualizar):
    sku = normalizar_sku(sku)

    producto = buscar_producto_por_sku(sku)

    if producto is None:
        return {
            "ok": False,
            "tipo_error": "no_encontrado",
            "mensaje": "Producto no encontrado."
        }

    campos = []
    valores = []

    if "nombre" in datos_actualizar and datos_actualizar["nombre"] is not None:
        nombre = validar_texto_obligatorio(datos_actualizar["nombre"], "nombre")
        campos.append("nombre = ?")
        valores.append(nombre)

    if "categoria" in datos_actualizar and datos_actualizar["categoria"] is not None:
        categoria = validar_texto_obligatorio(datos_actualizar["categoria"], "categoría")
        campos.append("categoria = ?")
        valores.append(categoria)

    if "precio" in datos_actualizar and datos_actualizar["precio"] is not None:
        precio = validar_precio(datos_actualizar["precio"])
        campos.append("precio = ?")
        valores.append(precio)

    if "stock" in datos_actualizar and datos_actualizar["stock"] is not None:
        stock = validar_stock(datos_actualizar["stock"])
        campos.append("stock = ?")
        valores.append(stock)

    if "activo" in datos_actualizar and datos_actualizar["activo"] is not None:
        activo = 1 if datos_actualizar["activo"] else 0
        campos.append("activo = ?")
        valores.append(activo)

    if not campos:
        return {
            "ok": False,
            "tipo_error": "validacion",
            "mensaje": "No se enviaron datos para actualizar."
        }

    valores.append(sku)

    sql = f"""
    UPDATE productos
    SET {", ".join(campos)}
    WHERE sku = ?
    """

    ejecutar(sql, tuple(valores))

    producto_actualizado = buscar_producto_por_sku(sku)

    return {
        "ok": True,
        "mensaje": "Producto actualizado correctamente.",
        "producto": producto_actualizado
    }


def desactivar_producto(sku):
    sku = normalizar_sku(sku)

    producto = buscar_producto_por_sku(sku)

    if producto is None:
        return {
            "ok": False,
            "tipo_error": "no_encontrado",
            "mensaje": "Producto no encontrado."
        }

    if producto["activo"] == 0:
        return {
            "ok": True,
            "mensaje": "El producto ya estaba inactivo.",
            "producto": producto
        }

    ejecutar("""
    UPDATE productos
    SET activo = 0
    WHERE sku = ?
    """, (sku,))

    producto_actualizado = buscar_producto_por_sku(sku)

    return {
        "ok": True,
        "mensaje": "Producto desactivado correctamente.",
        "producto": producto_actualizado
    }
'''

In [11]:
parte_services_7 = r'''


def listar_clientes(solo_activos=True):
    if solo_activos:
        return consultar_todos("""
        SELECT
            id_cliente,
            nombre,
            correo,
            telefono,
            ciudad,
            activo
        FROM clientes
        WHERE activo = 1
        ORDER BY nombre
        """)

    return consultar_todos("""
    SELECT
        id_cliente,
        nombre,
        correo,
        telefono,
        ciudad,
        activo
    FROM clientes
    ORDER BY nombre
    """)


def buscar_cliente_por_id(id_cliente):
    return consultar_uno("""
    SELECT
        id_cliente,
        nombre,
        correo,
        telefono,
        ciudad,
        activo
    FROM clientes
    WHERE id_cliente = ?
    """, (id_cliente,))


def registrar_cliente(nombre, correo, telefono=None, ciudad="No especificada"):
    try:
        nombre = validar_texto_obligatorio(nombre, "nombre")
        correo = normalizar_correo(correo)

        if correo == "":
            raise ValueError("El correo no puede estar vacío.")

        telefono = None if telefono is None else limpiar_texto(telefono)
        ciudad = limpiar_texto(ciudad) or "No especificada"

        resultado = ejecutar("""
        INSERT INTO clientes (nombre, correo, telefono, ciudad, activo)
        VALUES (?, ?, ?, ?, 1)
        """, (nombre, correo, telefono, ciudad))

        cliente = buscar_cliente_por_id(resultado["ultimo_id"])

        return {
            "ok": True,
            "mensaje": "Cliente registrado correctamente.",
            "cliente": cliente,
            "id_cliente": resultado["ultimo_id"]
        }

    except sqlite3.IntegrityError:
        return {
            "ok": False,
            "tipo_error": "conflicto",
            "mensaje": "Ya existe un cliente con ese correo."
        }

    except ValueError as error:
        return {
            "ok": False,
            "tipo_error": "validacion",
            "mensaje": str(error)
        }
'''

In [12]:
parte_services_8 = r'''


def registrar_venta(id_cliente, productos_vendidos):
    """
    Registra una venta completa.

    productos_vendidos:
    [
        {"sku": "P001", "cantidad": 2},
        {"sku": "P002", "cantidad": 1}
    ]
    """

    def operacion(cursor):
        cliente = buscar_cliente_por_id(id_cliente)

        if cliente is None:
            raise ValueError("No existe el cliente indicado.")

        if cliente["activo"] != 1:
            raise ValueError("El cliente está inactivo.")

        if not productos_vendidos:
            raise ValueError("No se puede registrar una venta sin productos.")

        fecha = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        cursor.execute("""
        INSERT INTO ventas (id_cliente, fecha, total, estado)
        VALUES (?, ?, 0, 'ACTIVA')
        """, (id_cliente, fecha))

        id_venta = cursor.lastrowid
        total_venta = 0

        for item in productos_vendidos:
            sku = normalizar_sku(item.get("sku"))
            cantidad = validar_cantidad(item.get("cantidad"))

            cursor.execute("""
            SELECT
                id_producto,
                sku,
                nombre,
                precio,
                stock,
                activo
            FROM productos
            WHERE sku = ?
            """, (sku,))

            producto = cursor.fetchone()

            if producto is None:
                raise ValueError(f"No existe el producto con SKU {sku}.")

            if producto["activo"] != 1:
                raise ValueError(f"El producto {sku} está inactivo.")

            if producto["stock"] < cantidad:
                raise ValueError(
                    f"Stock insuficiente para {sku}. "
                    f"Disponible: {producto['stock']}, solicitado: {cantidad}."
                )

            precio_unitario = producto["precio"]
            subtotal = precio_unitario * cantidad
            total_venta += subtotal

            cursor.execute("""
            INSERT INTO detalle_ventas (
                id_venta,
                id_producto,
                cantidad,
                precio_unitario,
                subtotal
            )
            VALUES (?, ?, ?, ?, ?)
            """, (
                id_venta,
                producto["id_producto"],
                cantidad,
                precio_unitario,
                subtotal
            ))

            cursor.execute("""
            UPDATE productos
            SET stock = stock - ?
            WHERE id_producto = ?
              AND stock >= ?
              AND activo = 1
            """, (
                cantidad,
                producto["id_producto"],
                cantidad
            ))

            if cursor.rowcount == 0:
                raise ValueError(f"No se pudo actualizar stock para {sku}.")

        cursor.execute("""
        UPDATE ventas
        SET total = ?
        WHERE id_venta = ?
        """, (total_venta, id_venta))

        return {
            "id_venta": id_venta,
            "total": total_venta
        }

    try:
        resultado = ejecutar_transaccion(operacion)

        return {
            "ok": True,
            "mensaje": "Venta registrada correctamente.",
            "id_venta": resultado["id_venta"],
            "total": resultado["total"]
        }

    except ValueError as error:
        return {
            "ok": False,
            "tipo_error": "validacion",
            "mensaje": str(error)
        }

    except sqlite3.IntegrityError:
        return {
            "ok": False,
            "tipo_error": "base_datos",
            "mensaje": "La venta viola una restricción de la base de datos."
        }
'''

In [13]:
parte_services_9 = r'''


def consultar_ticket(id_venta):
    return consultar_todos("""
    SELECT
        v.id_venta,
        v.fecha,
        v.estado,
        c.nombre AS cliente,
        c.correo,
        p.sku,
        p.nombre AS producto,
        dv.cantidad,
        dv.precio_unitario,
        dv.subtotal,
        v.total AS total_venta
    FROM ventas v
    INNER JOIN clientes c
        ON v.id_cliente = c.id_cliente
    INNER JOIN detalle_ventas dv
        ON v.id_venta = dv.id_venta
    INNER JOIN productos p
        ON dv.id_producto = p.id_producto
    WHERE v.id_venta = ?
    ORDER BY dv.id_detalle
    """, (id_venta,))


def listar_ventas():
    return consultar_todos("""
    SELECT
        v.id_venta,
        v.fecha,
        v.estado,
        c.nombre AS cliente,
        v.total
    FROM ventas v
    INNER JOIN clientes c
        ON v.id_cliente = c.id_cliente
    ORDER BY v.id_venta DESC
    """)


def resumen_ventas():
    resumen = consultar_uno("""
    SELECT
        COUNT(*) AS cantidad_ventas,
        COALESCE(SUM(total), 0) AS total_ingresos
    FROM ventas
    WHERE estado = 'ACTIVA'
    """)

    return resumen
'''

In [14]:
codigo_services = (
    parte_services_1
    + parte_services_2
    + parte_services_3
    + parte_services_4
    + parte_services_5
    + parte_services_6
    + parte_services_7
    + parte_services_8
    + parte_services_9
)

(PROYECTO_DIR / "services.py").write_text(codigo_services, encoding="utf-8")

print("services.py creado correctamente.")
print("Tamaño:", (PROYECTO_DIR / "services.py").stat().st_size, "bytes")

services.py creado correctamente.
Tamaño: 16376 bytes


In [15]:
parte_main_1 = r'''
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field
from typing import Optional, List

import services


app = FastAPI(
    title="API del Sistema de Ventas con SQLite",
    description="API conectada a SQLite para el curso Persistencia de Datos con Python.",
    version="2.0.0"
)


@app.on_event("startup")
def startup_event():
    services.inicializar_base_datos()
'''

In [16]:
parte_main_2 = r'''


class ProductoCrear(BaseModel):
    sku: str = Field(min_length=1)
    nombre: str = Field(min_length=1)
    categoria: str = Field(min_length=1)
    precio: float = Field(gt=0)
    stock: int = Field(ge=0)


class ProductoActualizar(BaseModel):
    nombre: Optional[str] = None
    categoria: Optional[str] = None
    precio: Optional[float] = Field(default=None, gt=0)
    stock: Optional[int] = Field(default=None, ge=0)
    activo: Optional[bool] = None


class ClienteCrear(BaseModel):
    nombre: str = Field(min_length=1)
    correo: str = Field(min_length=1)
    telefono: Optional[str] = None
    ciudad: Optional[str] = "No especificada"


class ItemVenta(BaseModel):
    sku: str = Field(min_length=1)
    cantidad: int = Field(gt=0)


class VentaCrear(BaseModel):
    id_cliente: int = Field(gt=0)
    productos: List[ItemVenta]


def modelo_a_dict(modelo, exclude_unset=False):
    if hasattr(modelo, "model_dump"):
        return modelo.model_dump(exclude_unset=exclude_unset)

    return modelo.dict(exclude_unset=exclude_unset)
'''

In [17]:
parte_main_3 = r'''


def manejar_resultado_servicio(resultado):
    """
    Convierte respuestas de services.py en respuestas HTTP.
    """
    if resultado.get("ok"):
        return resultado

    tipo_error = resultado.get("tipo_error")
    mensaje = resultado.get("mensaje", "Error en la operación.")

    if tipo_error == "no_encontrado":
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=mensaje
        )

    if tipo_error == "conflicto":
        raise HTTPException(
            status_code=status.HTTP_409_CONFLICT,
            detail=mensaje
        )

    if tipo_error == "validacion":
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail=mensaje
        )

    raise HTTPException(
        status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
        detail=mensaje
    )
'''

In [18]:
parte_main_4 = r'''


@app.get("/")
def inicio():
    return {
        "mensaje": "API conectada a SQLite funcionando.",
        "documentacion": "/docs",
        "version": "2.0.0"
    }


@app.get("/health")
def health_check():
    return {
        "status": "ok",
        "base_datos": "SQLite"
    }
'''

In [19]:
parte_main_5 = r'''


@app.get("/productos")
def listar_productos(solo_activos: bool = True):
    return services.listar_productos(solo_activos=solo_activos)


@app.get("/productos/{sku}")
def obtener_producto(sku: str):
    producto = services.buscar_producto_por_sku(sku)

    if producto is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Producto no encontrado."
        )

    return producto
'''

In [20]:
parte_main_6 = r'''


@app.post("/productos", status_code=status.HTTP_201_CREATED)
def crear_producto(datos: ProductoCrear):
    datos_producto = modelo_a_dict(datos)

    resultado = services.registrar_producto(
        sku=datos_producto["sku"],
        nombre=datos_producto["nombre"],
        categoria=datos_producto["categoria"],
        precio=datos_producto["precio"],
        stock=datos_producto["stock"]
    )

    return manejar_resultado_servicio(resultado)


@app.put("/productos/{sku}")
def actualizar_producto(sku: str, datos: ProductoActualizar):
    datos_actualizar = modelo_a_dict(datos, exclude_unset=True)

    resultado = services.actualizar_producto(
        sku=sku,
        datos_actualizar=datos_actualizar
    )

    return manejar_resultado_servicio(resultado)


@app.delete("/productos/{sku}")
def desactivar_producto(sku: str):
    resultado = services.desactivar_producto(sku)

    return manejar_resultado_servicio(resultado)
'''

In [21]:
parte_main_7 = r'''


@app.get("/clientes")
def listar_clientes(solo_activos: bool = True):
    return services.listar_clientes(solo_activos=solo_activos)


@app.get("/clientes/{id_cliente}")
def obtener_cliente(id_cliente: int):
    cliente = services.buscar_cliente_por_id(id_cliente)

    if cliente is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Cliente no encontrado."
        )

    return cliente


@app.post("/clientes", status_code=status.HTTP_201_CREATED)
def crear_cliente(datos: ClienteCrear):
    datos_cliente = modelo_a_dict(datos)

    resultado = services.registrar_cliente(
        nombre=datos_cliente["nombre"],
        correo=datos_cliente["correo"],
        telefono=datos_cliente.get("telefono"),
        ciudad=datos_cliente.get("ciudad")
    )

    return manejar_resultado_servicio(resultado)
'''

In [22]:
parte_main_8 = r'''


@app.post("/ventas", status_code=status.HTTP_201_CREATED)
def crear_venta(datos: VentaCrear):
    datos_venta = modelo_a_dict(datos)

    resultado = services.registrar_venta(
        id_cliente=datos_venta["id_cliente"],
        productos_vendidos=datos_venta["productos"]
    )

    return manejar_resultado_servicio(resultado)


@app.get("/ventas")
def listar_ventas():
    return services.listar_ventas()


@app.get("/ventas/resumen")
def resumen_ventas():
    return services.resumen_ventas()


@app.get("/ventas/{id_venta}/ticket")
def consultar_ticket(id_venta: int):
    ticket = services.consultar_ticket(id_venta)

    if not ticket:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Venta no encontrada."
        )

    return ticket
'''

In [23]:
codigo_main = (
    parte_main_1
    + parte_main_2
    + parte_main_3
    + parte_main_4
    + parte_main_5
    + parte_main_6
    + parte_main_7
    + parte_main_8
)

(PROYECTO_DIR / "main.py").write_text(codigo_main, encoding="utf-8")

print("main.py creado correctamente.")
print("Tamaño:", (PROYECTO_DIR / "main.py").stat().st_size, "bytes")

main.py creado correctamente.
Tamaño: 5873 bytes


In [25]:
BASE_URL = "http://127.0.0.1:8000"

In [26]:
def mostrar_respuesta(respuesta):
    print("Código de estado:", respuesta.status_code)

    try:
        print(json.dumps(respuesta.json(), indent=4, ensure_ascii=False))
    except Exception:
        print(respuesta.text)

In [27]:
respuesta = requests.get(f"{BASE_URL}/")
mostrar_respuesta(respuesta)

Código de estado: 200
{
    "mensaje": "API conectada a SQLite funcionando.",
    "documentacion": "/docs",
    "version": "2.0.0"
}


In [28]:
respuesta = requests.get(f"{BASE_URL}/productos")
mostrar_respuesta(respuesta)

Código de estado: 200
[
    {
        "id_producto": 6,
        "sku": "P006",
        "nombre": "Bocinas Bluetooth",
        "categoria": "Audio",
        "precio": 699.0,
        "stock": 10,
        "activo": 1
    },
    {
        "id_producto": 4,
        "sku": "P004",
        "nombre": "Cable HDMI",
        "categoria": "Cables",
        "precio": 120.0,
        "stock": 20,
        "activo": 1
    },
    {
        "id_producto": 5,
        "sku": "P005",
        "nombre": "Memoria USB 64GB",
        "categoria": "Almacenamiento",
        "precio": 150.0,
        "stock": 30,
        "activo": 1
    },
    {
        "id_producto": 3,
        "sku": "P003",
        "nombre": "Monitor 24 pulgadas",
        "categoria": "Pantallas",
        "precio": 3299.0,
        "stock": 4,
        "activo": 1
    },
    {
        "id_producto": 1,
        "sku": "P001",
        "nombre": "Mouse inalámbrico",
        "categoria": "Accesorios",
        "precio": 249.9,
        "stock": 15,
     

In [29]:
producto_nuevo = {
    "sku": "P007",
    "nombre": "Webcam HD",
    "categoria": "Accesorios",
    "precio": 550.00,
    "stock": 6
}

respuesta = requests.post(
    f"{BASE_URL}/productos",
    json=producto_nuevo
)

mostrar_respuesta(respuesta)

Código de estado: 201
{
    "ok": true,
    "mensaje": "Producto registrado correctamente.",
    "producto": {
        "id_producto": 7,
        "sku": "P007",
        "nombre": "Webcam HD",
        "categoria": "Accesorios",
        "precio": 550.0,
        "stock": 6,
        "activo": 1
    },
    "id_producto": 7
}


In [30]:
respuesta = requests.get(f"{BASE_URL}/productos/P007")
mostrar_respuesta(respuesta)

Código de estado: 200
{
    "id_producto": 7,
    "sku": "P007",
    "nombre": "Webcam HD",
    "categoria": "Accesorios",
    "precio": 550.0,
    "stock": 6,
    "activo": 1
}


In [31]:
respuesta = requests.put(
    f"{BASE_URL}/productos/P007",
    json={
        "precio": 600.00,
        "stock": 10
    }
)

mostrar_respuesta(respuesta)

Código de estado: 200
{
    "ok": true,
    "mensaje": "Producto actualizado correctamente.",
    "producto": {
        "id_producto": 7,
        "sku": "P007",
        "nombre": "Webcam HD",
        "categoria": "Accesorios",
        "precio": 600.0,
        "stock": 10,
        "activo": 1
    }
}


In [32]:
respuesta = requests.delete(f"{BASE_URL}/productos/P007")
mostrar_respuesta(respuesta)

Código de estado: 200
{
    "ok": true,
    "mensaje": "Producto desactivado correctamente.",
    "producto": {
        "id_producto": 7,
        "sku": "P007",
        "nombre": "Webcam HD",
        "categoria": "Accesorios",
        "precio": 600.0,
        "stock": 10,
        "activo": 0
    }
}


In [33]:
respuesta = requests.get(
    f"{BASE_URL}/productos",
    params={"solo_activos": False}
)

mostrar_respuesta(respuesta)

Código de estado: 200
[
    {
        "id_producto": 6,
        "sku": "P006",
        "nombre": "Bocinas Bluetooth",
        "categoria": "Audio",
        "precio": 699.0,
        "stock": 10,
        "activo": 1
    },
    {
        "id_producto": 4,
        "sku": "P004",
        "nombre": "Cable HDMI",
        "categoria": "Cables",
        "precio": 120.0,
        "stock": 20,
        "activo": 1
    },
    {
        "id_producto": 5,
        "sku": "P005",
        "nombre": "Memoria USB 64GB",
        "categoria": "Almacenamiento",
        "precio": 150.0,
        "stock": 30,
        "activo": 1
    },
    {
        "id_producto": 3,
        "sku": "P003",
        "nombre": "Monitor 24 pulgadas",
        "categoria": "Pantallas",
        "precio": 3299.0,
        "stock": 4,
        "activo": 1
    },
    {
        "id_producto": 1,
        "sku": "P001",
        "nombre": "Mouse inalámbrico",
        "categoria": "Accesorios",
        "precio": 249.9,
        "stock": 15,
     

In [34]:
respuesta = requests.get(f"{BASE_URL}/clientes")
mostrar_respuesta(respuesta)

Código de estado: 200
[
    {
        "id_cliente": 1,
        "nombre": "Ana López",
        "correo": "ana@example.com",
        "telefono": "555-111-2222",
        "ciudad": "CDMX",
        "activo": 1
    },
    {
        "id_cliente": 2,
        "nombre": "Carlos Pérez",
        "correo": "carlos@example.com",
        "telefono": "555-333-4444",
        "ciudad": "Guadalajara",
        "activo": 1
    },
    {
        "id_cliente": 4,
        "nombre": "Luis Romero",
        "correo": "luis@example.com",
        "telefono": "555-888-9999",
        "ciudad": "Puebla",
        "activo": 1
    },
    {
        "id_cliente": 3,
        "nombre": "María Torres",
        "correo": "maria@example.com",
        "telefono": null,
        "ciudad": "Monterrey",
        "activo": 1
    }
]


In [35]:
cliente_nuevo = {
    "nombre": "Sofía Herrera",
    "correo": "sofia@example.com",
    "telefono": None,
    "ciudad": "Querétaro"
}

respuesta = requests.post(
    f"{BASE_URL}/clientes",
    json=cliente_nuevo
)

mostrar_respuesta(respuesta)

Código de estado: 201
{
    "ok": true,
    "mensaje": "Cliente registrado correctamente.",
    "cliente": {
        "id_cliente": 5,
        "nombre": "Sofía Herrera",
        "correo": "sofia@example.com",
        "telefono": null,
        "ciudad": "Querétaro",
        "activo": 1
    },
    "id_cliente": 5
}


In [36]:
respuesta = requests.get(f"{BASE_URL}/clientes/5")
mostrar_respuesta(respuesta)

Código de estado: 200
{
    "id_cliente": 5,
    "nombre": "Sofía Herrera",
    "correo": "sofia@example.com",
    "telefono": null,
    "ciudad": "Querétaro",
    "activo": 1
}


In [37]:
venta_nueva = {
    "id_cliente": 1,
    "productos": [
        {
            "sku": "P001",
            "cantidad": 2
        },
        {
            "sku": "P002",
            "cantidad": 1
        }
    ]
}

respuesta = requests.post(
    f"{BASE_URL}/ventas",
    json=venta_nueva
)

mostrar_respuesta(respuesta)

Código de estado: 201
{
    "ok": true,
    "mensaje": "Venta registrada correctamente.",
    "id_venta": 1,
    "total": 1398.8
}


In [38]:
respuesta = requests.get(f"{BASE_URL}/ventas")
mostrar_respuesta(respuesta)

Código de estado: 200
[
    {
        "id_venta": 1,
        "fecha": "2026-06-25 18:48:23",
        "estado": "ACTIVA",
        "cliente": "Ana López",
        "total": 1398.8
    }
]


In [39]:
respuesta = requests.get(f"{BASE_URL}/ventas/1/ticket")
mostrar_respuesta(respuesta)

Código de estado: 200
[
    {
        "id_venta": 1,
        "fecha": "2026-06-25 18:48:23",
        "estado": "ACTIVA",
        "cliente": "Ana López",
        "correo": "ana@example.com",
        "sku": "P001",
        "producto": "Mouse inalámbrico",
        "cantidad": 2,
        "precio_unitario": 249.9,
        "subtotal": 499.8,
        "total_venta": 1398.8
    },
    {
        "id_venta": 1,
        "fecha": "2026-06-25 18:48:23",
        "estado": "ACTIVA",
        "cliente": "Ana López",
        "correo": "ana@example.com",
        "sku": "P002",
        "producto": "Teclado mecánico",
        "cantidad": 1,
        "precio_unitario": 899.0,
        "subtotal": 899.0,
        "total_venta": 1398.8
    }
]


In [40]:
respuesta = requests.get(f"{BASE_URL}/ventas/resumen")
mostrar_respuesta(respuesta)

Código de estado: 200
{
    "cantidad_ventas": 1,
    "total_ingresos": 1398.8
}


In [41]:
producto_duplicado = {
    "sku": "P001",
    "nombre": "Mouse duplicado",
    "categoria": "Accesorios",
    "precio": 100.00,
    "stock": 5
}

respuesta = requests.post(
    f"{BASE_URL}/productos",
    json=producto_duplicado
)

mostrar_respuesta(respuesta)

Código de estado: 409
{
    "detail": "Ya existe un producto con ese SKU."
}


In [42]:
respuesta = requests.get(f"{BASE_URL}/productos/NO_EXISTE")
mostrar_respuesta(respuesta)

Código de estado: 404
{
    "detail": "Producto no encontrado."
}


In [43]:
venta_stock_insuficiente = {
    "id_cliente": 1,
    "productos": [
        {
            "sku": "P003",
            "cantidad": 999
        }
    ]
}

respuesta = requests.post(
    f"{BASE_URL}/ventas",
    json=venta_stock_insuficiente
)

mostrar_respuesta(respuesta)

Código de estado: 400
{
    "detail": "Stock insuficiente para P003. Disponible: 4, solicitado: 999."
}
